# Building a new `cafpybara` analysis, file by file

This walks through every file in `analyses/_template/` (the directory this
notebook lives inside), in the order you'd actually write them, explaining
*why* each one is shaped the way it is -- not by abstract rule, but by
pointing at a real bug this project hit when that shape wasn't followed.
Every bug cited below is real and already fixed; this notebook exists so
the next analysis doesn't repeat them.

**Unlike the other example notebooks in this repo, this one needs no real
Fermilab/EAF data** -- `analyses/_template/` is self-contained and
importable on its own, so every cell here runs anywhere `cafpybara` itself
imports.

**What this notebook does NOT cover**: the actual mechanics of `CutSpec`/
`select()`/`modify_cut`/`drop_cuts`/`union_cut` -- that's already a
complete, hands-on walkthrough in
`analyses/hnlpi0/examples/selection_walkthrough.ipynb`, against real data.
This notebook covers everything *around* cuts: the file layout, why each
file exists, and the systematics/plotting wiring.

**To actually start a new analysis**: copy this whole `_template/`
directory to `analyses/<yourname>/`, work through the sections below in
order (each file only depends on the ones before it), and replace every
`# TODO` with real content for your topology.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("/exp/sbnd/data/users/lnguyen/cafpyana_pi0")

import pandas as pd
import cafpybara.analyses._template as ana
from cafpybara import core

print("Imported the template analysis cleanly -- it's a real, importable")
print("(if physically inert) analysis, not just documentation.")

## 1. `__init__.py` -- the re-export chain

This is the file most worth getting exactly right, unmodified, when you
copy the template. It does two things, in order:

1. `from ...core.X import *` for every `core` submodule -- pulls in every
   generic, topology-agnostic function/class.
2. `from .X import *` for this analysis's own `analysis`/`preprocess`/
   `io`/`plotting`/`funcs` -- these **shadow** the generic core versions
   with real, topology-specific defaults (e.g. `io.load_mc`'s real
   `rec_key`, where `core.io.load_mc` has none at all).

**Real bugs this exact pattern already caught, missing just one line
each**: `nuecc`'s and `hnlpi0`'s `__init__.py` were both independently
found missing `from ...core.io import *`/`core.funcs`/`core.plotting`/
`core.selection`/`core.physics` at various points -- each one a live
`AttributeError` (`nue.load_dfs`, `nue.integrated_flux`, etc. simply not
existing) the moment a notebook actually tried to use them. Nothing about
the missing line looked wrong on inspection; it only surfaced by trying to
use the missing name.

Check it for yourself below -- every one of these should resolve:

In [ ]:
core_modules_to_check = ['io', 'funcs', 'plotting', 'selection', 'physics', 'syst', 'classes', 'preprocess', 'detvar']
for mod_name in core_modules_to_check:
    core_mod = getattr(core, mod_name)
    missing = [n for n in getattr(core_mod, '__all__', []) if not hasattr(ana, n)]
    status = 'OK' if not missing else f'MISSING: {missing}'
    print(f"core.{mod_name:12s} -> {status}")

## 2. `config.py` -- your file paths

Just constants -- detvar store paths, in-time-cosmic file paths, anything
your analysis needs to know the location of.

**Design choice worth keeping**: the template's placeholders are
obviously-broken strings (`"TODO_SET_ME/..."`), not `None`. If code
actually tries to use an unfilled placeholder, you want a loud
`FileNotFoundError` immediately, not a `None` that might silently
propagate somewhere before failing (or not failing at all, and just
quietly doing the wrong thing) -- silently-wrong defaults are the single
biggest bug class this whole project has hit.

In [ ]:
print("DETVAR_DICT_FILES:", ana.config.DETVAR_DICT_FILES)
print("INTIME_FILE:", ana.config.INTIME_FILE)

## 3. `analysis.py` -- categories and cuts

Two things live here: your signal/background category dict (used for
stacked plots and the `signal` truth column), and your cut sequence.

**For the cuts half specifically, go work through
`analyses/hnlpi0/examples/selection_walkthrough.ipynb` now** -- it covers
`CutSpec`'s three forms, chaining cuts into a list, reading a cut-flow,
`modify_cut`/`drop_cuts` for extending an existing list, and `union_cut`
for "either A or B" selections, all hands-on against real data. Come back
here once you've got a real `DEFAULT_CUTS` for your topology.

The template's own versions are toy placeholders (illustrative column
names, not real ones) -- just enough to demonstrate the *shape* every
downstream file expects:

In [ ]:
print("signal_categories keys:", list(ana.signal_categories.keys()))
print("signal_dict:", ana.signal_dict)
print("\nDEFAULT_CUTS:")
for c in ana.DEFAULT_CUTS:
    print(f"  {c.name!r}: {c.label}")

## 4. `preprocess.py` -- the single most consequential file here

`core.preprocess.preprocess_mc`/`preprocess_data` are deliberately generic
no-ops. That's *correct* for a topology that genuinely needs no real
preprocessing -- but it is a real, live bug if your topology needs real
fixes (timing calibration, flash-PE rescaling, derived kinematics, ...)
and this file just quietly re-exports the no-op instead of building a
real composite on top of it.

**This exact mistake happened twice already**: nueCC's `load_mc`/
`load_data` were wired to `core`'s no-op instead of nueCC's own real
fixes -- silently affecting *every* MC/data DataFrame nueCC ever loaded,
only caught because a data plot showed 4 events where the known-good
baseline showed 39. Right after fixing that, an audit found the identical
pattern in `hnlpi0.load_mchnl`'s `preprocess_fn` default. Both are fixed
now, but neither was caught by static review -- only by noticing the
actual numbers were wrong.

The template's `preprocess_mc_real`/`preprocess_data_real` show the
pattern to extend -- start from the real base, add your own fixes, finish
with `add_variables`:

In [ ]:
import inspect
print(inspect.getsource(ana.preprocess.preprocess_mc_real))

## 5. `io.py` -- `load_mc`/`load_data` wrappers

Thin wrappers around `core.io.load_mc`/`load_data` that fill in this
analysis's real `rec_key`/`preprocess_fn`/`define_signal_fn`. Notice
`core.io.load_mc` itself has **no default** for any of these three --
that's deliberate, so a new analysis can't accidentally inherit another
topology's `rec_key` (nueCC's samples use `'nuecc'`, hnlpi0's use `'rec'`
-- there's no sensible shared default, so `core` doesn't try to guess).

In [ ]:
import inspect
sig = inspect.signature(core.io.load_mc)
print("core.io.load_mc's rec_key/preprocess_fn/define_signal_fn have no default:")
for name in ('rec_key', 'preprocess_fn', 'define_signal_fn'):
    p = sig.parameters[name]
    has_default = p.default is not inspect.Parameter.empty
    print(f"  {name}: {'default=' + repr(p.default) if has_default else 'no default (required)'}")
print("\nYour analysis's io.py fills these in -- REC_KEY placeholder:", ana.io.REC_KEY)

## 6. `funcs.py` -- `systs_input()`, and why there's no `get_total_cov` here

This is the file behind this session's biggest single redesign, so it's
worth understanding the actual failure it closes, not just the rule.

**The bug**: `core/plotting.py`'s `systs=<SystematicsInput>` handling
(what actually draws a plot's error band) used to call
`core.funcs.get_total_cov` *directly*. Meanwhile, notebook drill-down
cells called `nuecc.funcs.get_total_cov` -- a *different* function, which
happened to resolve nueCC's real defaults (like including `'cosmic'`
uncertainty) that the direct `core` call never saw. Both functions existed,
both looked reasonable, and they silently disagreed -- nueCC's actual
*plotted* error bands were missing an uncertainty source that the
drill-down tables right below the plot had. Nobody could tell just by
looking at the plot.

**The fix, and the rule that follows from it**: `core.funcs.get_total_cov`
is now the *only* `get_total_cov` anywhere -- no analysis defines its own.
Instead, each analysis provides `systs_input(...)`, a factory that fully
resolves every real default into a concrete `SystematicsInput` value
*before* anything reaches `core`. Both the plotted-band path and the
drill-down path now necessarily agree, because they consume the exact same
resolved object -- there's no second code path left for them to disagree
through.

In [ ]:
print("ana.get_total_cov IS core.funcs.get_total_cov:", ana.get_total_cov is core.funcs.get_total_cov)
print("(if this were ever False for a new analysis, that's the bug reintroduced)\n")

si = ana.systs_input(1e20, detvar_dict={'placeholder': True})
print("systs_input() resolved cuts to DEFAULT_CUTS:", si.cuts is ana.DEFAULT_CUTS)
print("resolved uncertainty_keys:", si.uncertainty_keys)
print("\nto_kwargs() ready to unpack into get_total_cov:")
print(list(si.to_kwargs().keys()))

## 7. `plotting.py` -- category injection

The same eager-resolution idea as `funcs.py`, applied to plot categories.
`core.plot_var` has **no** category default at all -- it raises a clear
error if it can't resolve one, rather than silently guessing. Your
analysis's `plot_var`/`plot_mc_data` wrappers fill in the real default
*before* calling into `core`, the same pattern as everywhere else in this
notebook.

In [ ]:
try:
    core.plotting.plot_var(df=pd.DataFrame(), var='x', bins=[0, 1])
except Exception as e:
    print(f"core.plot_var with no categories given raises immediately: {type(e).__name__}: {e}")

## 8. Before you ship it: a manual conformance pass

There's no automated checker for this yet (it's on the project to-do list
-- a script using this exact import-and-inspect technique, runnable with
no real data or Fermilab access, checking every point above
automatically). Until it exists, re-run the checks from sections 1 and 6
above against your real analysis module before considering it done:

In [ ]:
def conformance_check(analysis_module):
    """Run this against your real analysis module, not the template."""
    problems = []

    for mod_name in ['io', 'funcs', 'plotting', 'selection', 'physics', 'syst', 'classes', 'preprocess', 'detvar']:
        core_mod = getattr(core, mod_name)
        missing = [n for n in getattr(core_mod, '__all__', []) if not hasattr(analysis_module, n)]
        if missing:
            problems.append(f"core.{mod_name} not fully re-exported: missing {missing}")

    if analysis_module.get_total_cov is not core.funcs.get_total_cov:
        problems.append("get_total_cov has been overridden -- delete it, keep only systs_input()")

    if not hasattr(analysis_module, 'systs_input'):
        problems.append("no systs_input() factory found")

    if problems:
        print(f"{len(problems)} problem(s) found:")
        for p in problems:
            print(f"  - {p}")
    else:
        print("No problems found by this (non-exhaustive) check.")
    return problems


conformance_check(ana)

## Next steps

1. Copy `analyses/_template/` to `analyses/<yourname>/`.
2. Work through `analyses/hnlpi0/examples/selection_walkthrough.ipynb` to
   build your real `DEFAULT_CUTS` against real data.
3. Fill in every `# TODO` in `config.py`, `analysis.py`, `preprocess.py`,
   `io.py`, `funcs.py`, `plotting.py`, roughly in that order (each only
   depends on the ones before it).
4. Re-run this notebook's `conformance_check()` (section 8) against your
   real module, not the template.
5. See `README.md`'s "Reference: the override contract" section for the
   fuller per-file checklist, and "Common mistakes" for a couple of
   gotchas not covered above (e.g. `cuts=` needing to match between
   `load_mc` and `systs_input()`).